<a href="https://colab.research.google.com/github/redddddsu/Adaptative-SNN-Based-Triggering/blob/main/Software_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing API/Library


In [ ]:
!pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 1.0 MB/s eta 0:00:00


In [30]:
import torch
import torch.nn as nn
import snntorch as snn
import matplotlib.pyplot as plt
import os
import librosa
import numpy as np
from snntorch import surrogate

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Dataset

In [13]:

save_dir = "/content/drive/MyDrive/TUT_dataset"
os.makedirs(save_dir, exist_ok=True)

!wget -c -O /content/tut_source_events.zip \
"https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1"

!unzip -o /content/tut_source_events.zip -d /content/


Archive:  /content/tut_source_events.zip
   creating: /content/TUT-rare-sound-events-2017-development/
   creating: /content/TUT-rare-sound-events-2017-development/data/
   creating: /content/TUT-rare-sound-events-2017-development/data/source_data/
   creating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/
   creating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/107351.wav  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/107351.yaml  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/144473.wav  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/144473.yaml  
  inflating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/144474.wav  
  inflating: /content/TUT-rare-sound-ev

In [15]:
babycry_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry"

wav_files = sorted(f for f in os.listdir(babycry_path) if f.endswith(".wav"))
audio_path = os.path.join(babycry_path, wav_files[0])
audio, sr = librosa.load(audio_path, sr=None, mono=True)

# Mel Spectrogem

In [27]:
window_sample = round(sr * 0.064)
hops = round(sr * 0.032)

n_mel = 128
n_fft = window_sample

mel = librosa.feature.melspectrogram(y=audio,
    sr=sr,
    n_fft=n_fft,
    win_length=window_sample,
    hop_length=hops,
    n_mels=n_mel,
    power=2)


log_mel = librosa.power_to_db(mel, ref=np.max)
print(log_mel.shape)

# convert into frames (576)

(128, 576)


# Labeling Frames

In [28]:
import yaml

yaml_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry/107351.yaml"

with open(yaml_path, "r") as f:
    annotation = yaml.unsafe_load(f)

valid_segments = annotation["valid_segments"]
print(valid_segments)

start = valid_segments[0][0]
end = valid_segments[0][1]

print(start, end)

start_frame = round(start * sr / hops)
end_frame = round(end * sr / hops)

print("Start frame:", start_frame)
print("End frame:", end_frame)

labels = np.zeros(log_mel.shape[1])
labels[start_frame:end_frame] = 1

print(labels.shape)

[[0.18, 2.74]]
0.18 2.74
Start frame: 6
End frame: 86
(576,)


# Training

In [28]:

spike_grad = surrogate.fast_sigmoid()

class SNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(128, 128)
        self.lif1 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

        self.layer2 = nn.Linear(128, 128)
        self.lif2 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

        self.layer3 = nn.Linear(128, 127)
        self.lif3 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

        self.layer4 = nn.Linear(127, 1)
        self.lif4 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )
    def forward(self, x):
        potential1 = self.lif1.init_leaky()
        potentia2 = self.lif2.init_leaky()
        potentia3 = self.lif3.init_leaky()
        potentia4 = self.lif4.init_leaky()

        output = []

        for frame in x:
            spk1, potential1 = self.lif1(self.layer1(frame), potential1)
            spk2, potential2 = self.lif2(self.layer2(spk1), potential2)
            spk3, potential3 = self.lif3(self.layer3(spk2), potential3)
            spk4, potential4 = self.lif4(self.layer4(spk3), potential4)

            output.append(spk4)

        return torch.stack(output)
